In [ ]:
import os
from langgraph.graph import StateGraph, END
from ai_framework.nodes import GraphState , validate_question ,route_after_validation
from ai_framework.nodes import completedProcess,thinking_steps 

In [2]:
def build_graph():
    builder = StateGraph(GraphState)
    builder.add_node("validate", validate_question)
    builder.add_node("Thinking", thinking_steps)
    # builder.add_node("retrive_document", generate_answer)
    # builder.add_node("websearch", websearch)
    # builder.add_node("followupquestion", suggest_questions)
    builder.add_node("Completed", completedProcess)
    builder.set_entry_point("validate")
    builder.add_conditional_edges(
        "validate",
        route_after_validation,
        {
            "Thinking": "Thinking"
        }
    )

    builder.add_edge("Completed", END)

    return builder.compile()

In [3]:
# -----------------------------
# STREAM FUNCTION
# -----------------------------
def ask_question_stream(query: str):
    graph = build_graph()
    print("\n=== STREAMING START ===\n")
    final_state = {}
    for step in graph.stream({"query": query,"filename":"reactaa.pdf","messageId":"1234"}):
        for node, output in step.items():
            print(node, output)
            final_state.update(output)
    print("\n=== STREAMING END ===\n")
    return final_state


In [4]:
result=ask_question_stream("who is pm of india")
result


=== STREAMING START ===

validate {'is_valid': True}
state["query"] who is pm of india
Thinking {'thinking': {'content': '{\n    "Steps": [\n        "Step 1": "First, I will identify the type of question and determine the relevant knowledge domain.",\n        "Step 2": "Next, I will retrieve information from a reliable document or database using RAG to find the current Prime Minister of India.",\n        "Step 3": "Then, I will extract the relevant information from the retrieved data and verify its accuracy.",\n        "Step 4": "Finally, I will provide the answer in a', 'used_tokens': 100, 'prompt_tokens': 155, 'total_tokens': 255, 'messageid': '1234'}}

=== STREAMING END ===



{'is_valid': True,
 'thinking': {'content': '{\n    "Steps": [\n        "Step 1": "First, I will identify the type of question and determine the relevant knowledge domain.",\n        "Step 2": "Next, I will retrieve information from a reliable document or database using RAG to find the current Prime Minister of India.",\n        "Step 3": "Then, I will extract the relevant information from the retrieved data and verify its accuracy.",\n        "Step 4": "Finally, I will provide the answer in a',
  'used_tokens': 100,
  'prompt_tokens': 155,
  'total_tokens': 255,
  'messageid': '1234'}}

In [5]:
# prompt = template.replace("{query}",state["query"])
#     print(prompt) 